TD Final noté : Analyse des Avis et Alertes ANSSI avec
Enrichissement des CVE

● Extraire les données des flux RSS des avis et alertes ANSSI.

● Identifier les CVE mentionnées dans les bulletins.

● Enrichir les CVE avec des informations complémentaires via des API externes.


● Consolider les données dans un format exploitable (DataFrame pandas).


● Analyser et visualiser le DataFrame obtenu (vulnérabilités critiques, scores...)


● Modèles Machine learning


● Générer des alertes personnalisées pour les produits affectés et envoyer des notifications par email.

## Étape 1 — Extraction des flux RSS ANSSI

Les avis et alertes du CERT-FR sont récupérés via la fonction réutilisable
`anssi.feeds.fetch_all_feeds`. Chaque entrée est normalisée en un objet
`BulletinEntry` : **identifiant** (`CERTFR-...`), **type** (Avis / Alerte),
**titre**, **description**, **date de publication** (UTC) et **lien**.
La fonction est robuste aux champs manquants, normalise l'encodage et les
dates, et dédoublonne les bulletins.

Deux sources produisent le **même** schéma :

- **`source="rss"`** — le flux RSS live du CERT-FR ; ne renvoie que les
  bulletins les plus récents. C'est l'entrée décrite par le sujet.
- **`source="local"`** — les JSON pré-téléchargés (`data/data/`), qui couvrent
  **l'historique complet** (≈ 4 100 bulletins depuis 2021). C'est la base de
  travail des étapes d'analyse et de ML.

> **Accès responsable** (§8 du sujet) : en mode RSS, le flux brut est mis en
> cache (`data/feeds_cache/`) et relu au lieu d'être re-téléchargé, avec un
> délai de 2 s entre deux requêtes réseau.

In [ ]:
from anssi.feeds import fetch_all_feeds, feed_to_dataframe

# Flux RSS live (derniers bulletins) — la source décrite par le sujet.
recents = fetch_all_feeds(source="rss")
print(f"Flux RSS  : {len(recents)} bulletins récents")

# Corpus complet (JSON pré-téléchargés) — base de travail pour l'analyse.
bulletins = fetch_all_feeds(source="local")
print(f"Corpus local : {len(bulletins)} bulletins (avis + alertes)")

df_bulletins = feed_to_dataframe(bulletins)
df_bulletins.head()

## Étape 3 — Enrichissement des CVE (MITRE + FIRST/EPSS)

Chaque CVE est enrichi via deux sources :

| Source | Données | Particularités gérées |
|---|---|---|
| **MITRE** | Description, CVSS, CWE, produits/versions | 54 % sans CVSS ; priorité `v4.0 > v3.1 > v3.0 > v2.0` ; `type="cwe"` (minuscule) ; `metrics: null` ; versions avec `lessThan`/`lessThanOrEqual` |
| **FIRST** | Score EPSS, percentile | Valeurs manquantes → `None` |

Toutes les valeurs absentes sont `None` (jamais NaN) pour compatibilité pandas.

In [ ]:
from anssi.enrichment import enrich_cves, enrichment_to_dataframe

# Liste des CVE à enrichir (issus de l'étape 2)
# all_cve_ids = list({cve for b in bulletin_cves for cve in b.cves})

# Exemple sur un sous-ensemble pour démonstration
sample_ids = ["CVE-2024-20919", "CVE-2016-8625", "CVE-2026-40164", "CVE-2020-36187"]
enrichments = enrich_cves(sample_ids, source="local")

for cve_id, e in enrichments.items():
    m = e.mitre
    print(f"{cve_id}  CVSS v{m.cvss_version} {m.cvss_score} ({m.cvss_severity})  "
          f"CWE={m.cwe_id}  EPSS={e.first.epss_score}")

df_enrichment = enrichment_to_dataframe(enrichments)
df_enrichment